# DACON ETRI S3 raw-sequence random-conv remote sweep runner

Memory-safe Kaggle runner for report-only S3 hard-wall random-convolution sequence diagnostics. The local Mac should only package/push/pull; raw parquet expansion and repeated model fitting happen on Kaggle. No DACON submission is created by this runner. The goal is to find a genuinely different S3 representation after local compact HR/BLE/screen/hourly features saturated.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

def run(cmd: str) -> None:
    print(f'\n$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)

is_kaggle = Path('/kaggle/input').exists()
work_dir = Path('/kaggle/working') if is_kaggle else Path.cwd()
project_root = Path('/tmp/dacon_project') if is_kaggle else Path.cwd()
output_root = work_dir / 'kaggle_outputs' if is_kaggle else Path('kaggle_outputs')
os.environ['KAGGLE_OUTPUT_DIR'] = str(output_root)

required_payload_paths = [
    Path('scripts/diagnose_public_random_conv_sequence.py'),
    Path('artifacts/base_predictions/base_oof_id_247_id235_frontier_twise_cap020_logit_bridge_cv0.415523.csv'),
    Path('artifacts/base_predictions/base_test_id_247_id235_frontier_twise_cap020_logit_bridge_cv0.415523.csv'),
]

def payload_score(root: Path) -> int:
    return sum((root / rel).exists() for rel in required_payload_paths)

if is_kaggle:
    print('Detected Kaggle environment; locating project payload...')
    source_zips = sorted(Path('/kaggle/input').glob('**/source.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    project_root.mkdir(parents=True, exist_ok=True)
    if source_zips:
        with zipfile.ZipFile(source_zips[0]) as zf:
            zf.extractall(project_root)
        print(f'Extracted {source_zips[0]} -> {project_root}')
    else:
        payload_roots = []
        for root, dirs, files in os.walk('/kaggle/input'):
            dir_set = set(dirs)
            if {'src', 'conf', 'scripts'}.issubset(dir_set):
                payload_roots.append(Path(root))
        if not payload_roots:
            raise FileNotFoundError('No source.zip or unpacked project root with src/conf/scripts found in Kaggle inputs.')
        scored = [(payload_score(root), root) for root in payload_roots]
        for score, root in scored:
            print(f'Payload candidate score={score}: {root}')
        payload_root = sorted(payload_roots, key=lambda x: (-payload_score(x), len(str(x))))[0]
        print(f'Found unpacked payload at: {payload_root}')
        for name in ['src', 'conf', 'scripts', 'data', 'artifacts', 'submissions', 'pyproject.toml', 'README.md']:
            src_path = payload_root / name
            dst_path = project_root / name
            if not src_path.exists():
                continue
            if src_path.is_dir():
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_path, dst_path)
        print(f'Copied unpacked payload -> {project_root}')
else:
    print('Local dry-run mode; using current repository root.')

os.chdir(project_root)
sys.path.insert(0, str(project_root))
for folder in ['data/processed', 'reports', 'submissions', 'artifacts/base_predictions', 'notebooks/history']:
    Path(folder).mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
missing_required = [str(path) for path in required_payload_paths if not path.exists()]
if missing_required:
    print('Top-level project files:', sorted(str(p) for p in Path('.').iterdir())[:40])
    print('scripts sample:', sorted(str(p) for p in Path('scripts').glob('*.py'))[:40] if Path('scripts').exists() else [])
    raise FileNotFoundError(f'Missing required S3 random-conv payload files: {missing_required}')
print('project_root =', project_root)
print('output_root =', output_root)
print('required payload files verified:', [str(path) for path in required_payload_paths])



In [ ]:
# Keep installs explicit and CPU-friendly. Kaggle usually has most of these already; install only missing packages.
if is_kaggle:
    import importlib.util

    required = {
        'lightgbm': 'lightgbm',
        'xgboost': 'xgboost',
        'catboost': 'catboost',
        'nbformat': 'nbformat',
        'holidays': 'holidays',
        'pyarrow': 'pyarrow',
        'fastparquet': 'fastparquet',
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    print("missing packages =", missing)
    if missing:
        run(f"{sys.executable} -m pip install -q {' '.join(missing)}")


In [ ]:
# Build processed caches on Kaggle from raw parquet if they are not bundled.
from pathlib import Path
import pandas as pd

train_cache = Path('data/processed/df_train_raw.parquet')
test_cache = Path('data/processed/df_test_raw.parquet')
if train_cache.exists() and test_cache.exists():
    print('Processed caches already exist; skipping rebuild.')
else:
    from src.data.make_dataset import build_training_dataset

    raw_dir = Path('data/raw')
    sensor_dir = raw_dir / 'ch2025_data_items'
    if not sensor_dir.exists():
        raise FileNotFoundError(f'Missing raw sensor dir: {sensor_dir}')
    sensor_data = {}
    for parquet_path in sorted(sensor_dir.glob('*.parquet')):
        print('Loading', parquet_path)
        sensor_data[parquet_path.stem] = pd.read_parquet(parquet_path)
    metrics = pd.read_csv(raw_dir / 'ch2026_metrics_train.csv')
    sample = pd.read_csv(raw_dir / 'ch2026_submission_sample.csv')
    train_df = build_training_dataset(metrics, sensor_data)
    test_df = build_training_dataset(sample, sensor_data)
    train_df.to_parquet(train_cache, index=False)
    test_df.to_parquet(test_cache, index=False)
    print('Wrote processed caches:', train_df.shape, test_df.shape)


In [ ]:
# Report-only remote S3 random-conv sweep. Keep this sequential to cap memory.
# This runner intentionally uses ID247 as the anchor because its S3 column is the current preserved S3 base.
# If any report shows S3 improvement, pull outputs locally and promote via the normal unique ID/log/notebook/report contract.
import time
RUN_STARTED_TS = time.time()
SWEEP_JOBS = [
    {"base_id": 247, "n_kernels": 512, "seed": 45201, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk512_seed45201.json"},
    {"base_id": 247, "n_kernels": 512, "seed": 45202, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk512_seed45202.json"},
    {"base_id": 247, "n_kernels": 512, "seed": 45203, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk512_seed45203.json"},
    {"base_id": 247, "n_kernels": 768, "seed": 45301, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk768_seed45301.json"},
    {"base_id": 247, "n_kernels": 768, "seed": 45302, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk768_seed45302.json"},
    {"base_id": 247, "n_kernels": 1024, "seed": 45401, "output": "reports/id247_s3_random_conv_sequence_diagnostic_nk1024_seed45401.json"},
]
STRICT_EXPERIMENT_COMMAND = (
    f"S3 hard-wall raw random-conv sequence sweep: {len(SWEEP_JOBS)} jobs; "
    f"base_ids={sorted({job['base_id'] for job in SWEEP_JOBS})}; "
    f"kernels={sorted({job['n_kernels'] for job in SWEEP_JOBS})}; "
    f"seeds={[job['seed'] for job in SWEEP_JOBS]}"
)
print(STRICT_EXPERIMENT_COMMAND)
for job in SWEEP_JOBS:
    output_path = Path(job['output'])
    staged_path = output_root / output_path.name
    if output_path.exists():
        print(f"Skipping existing output: {output_path}")
    else:
        cmd = (
            f"{sys.executable} scripts/diagnose_public_random_conv_sequence.py "
            f"--base-id {job['base_id']} "
            f"--n-kernels {job['n_kernels']} "
            f"--seed {job['seed']} "
            f"--output {output_path}"
        )
        run(cmd)
    if output_path.exists():
        shutil.copy2(output_path, staged_path)
        print(f"Staged {output_path} -> {staged_path}")


In [ ]:
# Stage small artifacts for `kaggle kernels output`.
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

out_dir = Path(os.environ.get('KAGGLE_OUTPUT_DIR', 'kaggle_outputs'))
out_dir.mkdir(parents=True, exist_ok=True)
patterns = [
    'reports/id*_random_conv_sequence_diagnostic*.json',
    'reports/strict_*diagnostic*.json',
    'reports/experiment_*.json',
    'submissions/submission_id_*.csv',
    'submissions/submission_log.csv',
    'artifacts/base_predictions/base_oof_id_*.csv',
    'artifacts/base_predictions/base_test_id_*.csv',
    'notebooks/history/submission_*.ipynb',
]
copied = []
run_started_ts = float(globals().get('RUN_STARTED_TS', 0.0))
for pattern in patterns:
    for path in sorted(Path('.').glob(pattern), key=lambda p: p.stat().st_mtime):
        if not path.is_file():
            continue
        if path.stat().st_mtime < run_started_ts - 5.0:
            continue
        dest = out_dir / path.name
        shutil.copy2(path, dest)
        copied.append(str(path))
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'command': STRICT_EXPERIMENT_COMMAND,
    'sweep_jobs': globals().get('SWEEP_JOBS', []),
    'copied_artifacts': copied,
    'note': 'Pull these S3 random-conv outputs locally, review diagnostics, then promote with a unique ID/log/notebook/report/base-prediction contract only if S3 improves.',
}
(out_dir / 'remote_strict_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))

